# RAG Example: Loading Your Own Data into ChromaDB

This notebook shows how to load **any text data** into ChromaDB and query it.

We'll use the famous "Wear Sunscreen" speech (Chicago Tribune, 1997) as our example dataset.

Original source: [Salem Public Library — 1999 Yearbook](http://history.salem.lib.oh.us/yearbooks/1999/2facesplits/1999op_Part9.pdf)

## Step 1 — Read and chunk the text

The text file has the speech split into paragraphs (separated by blank lines).
We split on `\n\n` so each paragraph becomes one chunk in our database.

In [1]:
with open("../data/tarot_database_rag.txt") as f:
    text = f.read()

chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print(f"{len(chunks)} chunks loaded\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk[:80]}...\n")

443 chunks loaded

Chunk 0: Tarot Card: The Fool
Number: 0
Arcana: Major Arcana...

Chunk 1: Fortune Telling:
Watch for new projects and new beginnings | Prepare to take som...

Chunk 2: Light Meanings:
Freeing yourself from limitation | Expressing joy and youthful v...

Chunk 3: Shadow Meanings:
Being gullible and naive | Taking unnecessary risks | Failing t...

Chunk 4: Questions to Ask:...

Chunk 5: ---...

Chunk 6: Tarot Card: The Magician
Number: 1
Arcana: Major Arcana...

Chunk 7: Fortune Telling:
A powerful man may play a role in your day | Your current situa...

Chunk 8: Light Meanings:
Taking appropriate action | Receiving guidance from a higher pow...

Chunk 9: Shadow Meanings:
Inflating your own ego | Abusing talents | Manipulating or dece...

Chunk 10: Questions to Ask:...

Chunk 11: ---...

Chunk 12: Tarot Card: The High Priestess,2,Major Arcana,A mysterious woman arrives | A sex...

Chunk 13: Fortune Telling:
nan...

Chunk 14: Light Meanings:
nan...

Chunk 15: Shadow Mean

## Step 2 — Load into ChromaDB

In [2]:
import chromadb

client = chromadb.PersistentClient("../chroma")

# Delete collection if it already exists (so we can re-run this cell)
try:
    client.delete_collection("tarot_cards_rag")
except Exception:
    pass

collection = client.create_collection(name="tarot_cards_rag")

collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)

print(f"Added {collection.count()} chunks to the 'tarot_cards_rag' collection")

Added 443 chunks to the 'tarot_cards_rag' collection


## Step 3 — Query the collection

ChromaDB uses the query text to find the most semantically similar chunks.
This is the same pattern we use in `5_rag.ipynb` with the nutrition database.

In [3]:
results = collection.query(
    query_texts=["What does the Emperor card represent?"],
    n_results=3,
)

for i, doc in enumerate(results["documents"][0]):
    print(f"Result {i + 1}:")
    print(doc)
    print()

Result 1:
Tarot Card: The Empress
Number: 3
Arcana: Major Arcana

Result 2:
Tarot Card: The Chariot
Number: 7
Arcana: Major Arcana

Result 3:
Tarot Card: The Hermit
Number: 9
Arcana: Major Arcana



In [4]:
# We populated the RAG with the data from the data/calories.csv file in
# the ../rag_setup/rag_setup.ipynb notebook

chroma_client = chromadb.PersistentClient("../chroma")
tarot_cards_rag = chroma_client.get_collection(name="tarot_cards_rag")